# Correlation Tutorial 4: MLIP Topological Descriptors & Structural-Electronic Correlation

In modern computational materials science, Machine Learning Interatomic Potentials (MLIPs) and Graph Neural Networks (GNNs) bridge structural topology and electronic structure.

This tutorial demonstrates:
1. Constructing periodic neighbor graphs from atomic configurations (`PeriodicGraphData`).
2. Extracting topological graph descriptors (coordination embeddings, cycle-basis ring statistics, graph spectral eigenvalues, and CNA structural signatures).
3. Defining a custom electronic structure model using the `correlation.MLIPInterface` Python bridge.
4. Correlating per-atom Local Density of States (LDoS) with local structural motifs (`correlate_cna` and `correlate_steinhardt`).
5. Visualizing the **Motif-Projected Total Density of States (TDOS)** and verifying electronic conservation.


## 1. Imports and Verification


In [ ]:
import correlation
import numpy as np
import matplotlib.pyplot as plt

print("Correlation Version:", getattr(correlation, "__version__", "4.0.0"))
print("Calculators registered:", len(correlation.list_calculators()))


## 2. Generating a Multi-Phase / Disordered Trajectory

We create a representative copper (Cu) lattice (FCC crystal parameter $a = 3.615$ Å) containing both pristine crystalline regions and thermally disordered/distorted local environments.


In [ ]:
# Create an FCC supercell (2x2x2 unit cells = 32 atoms)
a = 3.615
cell = correlation.Cell([a * 2, a * 2, a * 2, 90.0, 90.0, 90.0])

fcc_basis = np.array([
    [0.0, 0.0, 0.0],
    [0.0, 0.5, 0.5],
    [0.5, 0.0, 0.5],
    [0.5, 0.5, 0.0]
])

rng = np.random.default_rng(seed=42)
traj = correlation.Trajectory()

# Synthesize a short 5-frame trajectory with thermal fluctuations
for frame_idx in range(5):
    frame_cell = correlation.Cell([a * 2, a * 2, a * 2, 90.0, 90.0, 90.0])
    atom_idx = 0
    for ix in range(2):
        for iy in range(2):
            for iz in range(2):
                origin = np.array([ix, iy, iz])
                for b in fcc_basis:
                    pos = (origin + b) * a
                    # Introduce significant disorder into half the atoms (simulating a grain boundary or defect)
                    if atom_idx >= 16:
                        pos += rng.normal(0.0, 0.25, size=3)
                    else:
                        pos += rng.normal(0.0, 0.04, size=3)
                    frame_cell.add_atom("Cu", pos.tolist())
                    atom_idx += 1
    traj.add_frame(frame_cell)

print(f"Trajectory created with {len(traj)} frames, {traj[0].atom_count} atoms per frame.")


## 3. Extracting Topological Graph Descriptors

Correlation provides high-performance graph construction directly in C++ via `build_periodic_graph` and `populate_descriptors`.
- **Coordination Embeddings**: Exact degree of each atom in the periodic neighbor network.
- **Ring Statistics**: Multi-scale cycle-basis ring counts (e.g. 3- to 6-membered rings).
- **Common Neighbor Analysis (CNA)**: Faken-Andersen local motif classifications (FCC, HCP, BCC, icosahedral, or disordered/other).
- **Graph Spectral Decomposition**: Top-$k$ eigenvalues of the graph adjacency matrix characterizing overall network connectivity.


In [ ]:
sample_cell = traj[0]
cutoff = 2.9  # Nearest-neighbor cutoff in Å for Cu FCC lattice

# Build periodic neighbor graph tensor buffers
graph = correlation.build_periodic_graph(sample_cell, cutoff)
print(f"Graph constructed: {graph.atom_count} atoms, {graph.edge_count} directed edges.")

# Populate all descriptors in-place (up to ring size 6)
correlation.populate_descriptors(graph, max_ring_size=6)

# Inspect topological features
coords = graph.coordination_desc
cna = graph.cna_labels
rings = graph.ring_desc

print(f"Mean coordination number: {np.mean(coords):.2f} (pristine FCC: 12.0)")
print(f"Ring tensor shape: {rings.shape} (Atoms x Ring Sizes 1..6)")

# Compute top-5 graph spectral eigenvalues
spectrum = correlation.compute_graph_spectrum(graph, k=5)
print(f"Top-5 Adjacency Eigenvalues: {[round(v, 4) for v in spectrum]}")


## 4. Defining a Python MLIP Electronic Structure Model

With the `correlation.MLIPInterface` trampoline, you can integrate PyTorch GNNs or custom Python surrogate models. Here we define an electronic model that predicts a per-atom Local Density of States (LDoS) profile whose bandwidth and shape depend on whether the atom is in a pristine crystalline or disordered coordination environment.


In [ ]:
class ElectronicStructureSurrogate(correlation.MLIPInterface):
    def __init__(self, num_bins=80, e_min=-8.0, e_max=4.0):
        super().__init__()
        self.num_bins = num_bins
        self.e_min = e_min
        self.e_max = e_max
        self.energies = np.linspace(e_min, e_max, num_bins)

    def get_model_name(self):
        return "GNN-Surrogate-Cu-d-band"

    def evaluate(self, cell):
        output = correlation.MLIPOutput()
        n = cell.atom_count
        output.ldos_bins = self.num_bins

        # Construct per-atom LDOS:
        # Atoms with ordered FCC local coordination exhibit sharp d-band resonance around -2.5 eV.
        # Disordered atoms exhibit broadened, defect-smeared distributions.
        ldos_matrix = []
        for i in range(n):
            if i < 16:  # Crystalline region
                # Sharp peak at -2.5 eV
                peak = np.exp(-0.5 * ((self.energies - (-2.5)) / 0.6) ** 2)
            else:        # Disordered region
                # Broadened, shifted peak at -2.0 eV with shoulder
                peak = np.exp(-0.5 * ((self.energies - (-2.0)) / 1.3) ** 2) + 0.2 * np.exp(-0.5 * ((self.energies - (-0.5)) / 0.8) ** 2)
            
            # Normalize to 10 electrons per atom
            peak = (peak / np.sum(peak)) * 10.0
            ldos_matrix.append(peak.tolist())

        output.ldos = ldos_matrix
        return output

model = ElectronicStructureSurrogate(num_bins=80, e_min=-8.0, e_max=4.0)
print(f"Initialized electronic model: {model.get_model_name()}")


## 5. Structural-Electronic Correlation Pipeline

We now project the predicted Local Density of States onto local structural motifs across the entire trajectory using `correlation.correlate_cna` and `correlation.correlate_steinhardt`.


In [ ]:
# Initialize DistributionFunctions container and TDOS parameters
dists = correlation.DistributionFunctions(traj[0])
params = correlation.TDOSParams(e_min=-8.0, e_max=4.0, model=model)

# Correlate trajectory LDoS with Common Neighbor Analysis motifs
motif_cna = correlation.correlate_cna(dists, traj, params)

print("Motif-Projected TDOS (CNA):")
print(f"  Energy grid: {len(motif_cna.energies)} bins from {motif_cna.energies[0]:.2f} to {motif_cna.energies[-1]:.2f} eV")
print(f"  Detected structural motifs: {list(motif_cna.motif_tdos.keys())}")
print(f"  Evaluated frames: {motif_cna.frame_count}")


## 6. Verifying Electronic Conservation

A fundamental physical invariant of motif-projected density of states is **spectral conservation**:
$$\text{TDOS}_{\text{total}}(E) = \sum_{m \in \text{motifs}} \text{TDOS}_{m}(E)$$
We numerically verify this conservation across all energy bins.


In [ ]:
energies = np.array(motif_cna.energies)
total_tdos = np.array(motif_cna.total_tdos)

# Sum all motif partials
sum_partials = np.zeros_like(total_tdos)
for motif_name, partial in motif_cna.motif_tdos.items():
    sum_partials += np.array(partial)

max_residual = np.max(np.abs(total_tdos - sum_partials))
print(f"Maximum conservation residual |Total - Sum(Motifs)|: {max_residual:.2e}")
assert max_residual < 1e-5, "Conservation check failed!"
print("✓ Electronic conservation strictly verified!")


## 7. Visualizing the Motif-Projected Electronic Structure

We plot the decomposed Total Density of States (TDOS) to reveal how local structural motifs contribute to the macroscopic electronic spectrum.


In [ ]:
plt.style.use("seaborn-v0_8-whitegrid" if "seaborn-v0_8-whitegrid" in plt.style.available else "default")
fig, ax = plt.subplots(figsize=(9, 5), dpi=120)

# Plot total TDOS
ax.plot(energies, total_tdos, label="Total TDOS", color="#111827", lw=2.2, zorder=4)

# Color map for motifs
motif_colors = {
    "FCC": "#0284c7",
    "HCP": "#10b981",
    "BCC": "#f59e0b",
    "ICO": "#8b5cf6",
    "Other": "#ef4444",
    "Disordered": "#ef4444"
}

# Plot partial motif curves
for motif_name, partial in motif_cna.motif_tdos.items():
    p_arr = np.array(partial)
    color = motif_colors.get(motif_name, "#6b7280")
    ax.plot(energies, p_arr, label=f"{motif_name} Motif Partial", color=color, lw=1.6, ls="--", zorder=3)
    ax.fill_between(energies, p_arr, alpha=0.15, color=color)

ax.axvline(0.0, color="#9ca3af", ls=":", lw=1.2, label="Fermi Level ($E_F$)")
ax.set_xlabel("Energy $E - E_F$ (eV)", fontsize=12, fontweight="bold")
ax.set_ylabel("Density of States (states/eV/atom)", fontsize=12, fontweight="bold")
ax.set_title("Structural-Electronic Correlation: Motif-Projected TDOS (Cu Lattice)", fontsize=13, fontweight="bold", pad=12)
ax.legend(loc="upper right", frameon=True, framealpha=0.9)
ax.set_xlim(motif_cna.energies[0], motif_cna.energies[-1])
ax.set_ylim(bottom=0.0)
plt.tight_layout()
plt.show()


## Summary

In this tutorial, you learned how to:
- Construct periodic graphs and extract multi-scale topological descriptors (`GraphDescriptors`).
- Bridge custom Python ML models with Correlation via `correlation.MLIPInterface`.
- Partition trajectory-level electronic Density of States into structural motifs via `correlate_cna` and `correlate_steinhardt`.
- Numerically verify electronic conservation in motif-projected spectra.
